# 5단계: Stacking_LR 임계값 민감도 확인

> 이전 실험 기록: 아래 코드와 출력은 이전 4단계 Stacking의 임계값 검토다. 최신 RF 앙상블은 임계값 0.5를 유지하며 이 노트북을 사용하지 않는다. 현재 실행 순서는 [노트북 안내](README.md)를 따른다.

4단계에서 저장한 반복 OOF 확률만 사용해 분류 임계값을 `0.5000` 주변에서 미세하게 비교한다.

- 비교 범위: `0.4800`부터 `0.5200`까지
- 비교 간격: `0.0025`
- 모델과 예측 확률은 고정하고 분류 판정 기준만 변경
- `0.5000` 유지 여부는 반복 Group CV 결과로 판단
- Test 결과로 임계값을 다시 선택하지 않음

Brier와 AUC는 임계값에 따라 변하지 않으므로 이 단계에서는 Accuracy·Precision·Recall·F1·FP·FN의 변화를 비교한다.

## 0. 실행 순서

4단계 노트북을 끝까지 실행해 `deal_phase4_predictions.joblib`을 만든 뒤 실행한다.

```bash
uv run --project backend/notebooks --locked jupyter lab backend/notebooks/deal_model_phase5_threshold.ipynb
```

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 260)

current_dir = Path.cwd().resolve()
if current_dir.name == "notebooks":
    repo_root = current_dir.parents[1]
elif current_dir.name == "backend" and (current_dir / "notebooks").is_dir():
    repo_root = current_dir.parent
elif (current_dir / "backend" / "notebooks").is_dir():
    repo_root = current_dir
else:
    raise RuntimeError("backend/notebooks 폴더를 찾을 수 없습니다.")

PHASE4_PREDICTION_PATH = (
    repo_root / "backend" / "pipeline" / "artifacts" / "deal_phase4_predictions.joblib"
)
assert PHASE4_PREDICTION_PATH.exists(), (
    "4단계 확률 아티팩트가 없습니다. deal_model_phase4.ipynb를 끝까지 실행하세요: "
    f"{PHASE4_PREDICTION_PATH}"
)

phase4_artifact = joblib.load(PHASE4_PREDICTION_PATH)
assert phase4_artifact["schema_version"] == 1
assert phase4_artifact["selected_candidate"] == "Stacking_LR"

cv_probabilities = np.asarray(phase4_artifact["cv_probabilities"], dtype=float)
cv_target = np.asarray(phase4_artifact["cv_target"], dtype=int)
cv_mask_set_labels = np.asarray(phase4_artifact["cv_mask_set_labels"])
BASELINE_THRESHOLD = float(phase4_artifact["classification_threshold"])
THRESHOLDS = np.round(np.arange(0.4800, 0.5200 + 0.0001, 0.0025), 4)

assert cv_probabilities.ndim == 2
assert cv_probabilities.shape[1] == len(cv_target) == len(cv_mask_set_labels)
assert np.isfinite(cv_probabilities).all()
assert ((cv_probabilities >= 0) & (cv_probabilities <= 1)).all()
assert BASELINE_THRESHOLD in THRESHOLDS

print(f"선택 모델: {phase4_artifact['selected_candidate']}")
print(f"반복 OOF 확률: {cv_probabilities.shape}")
print(f"마스킹 세트: {len(np.unique(cv_mask_set_labels))}개")
print(
    f"임계값: {THRESHOLDS[0]:.4f} ~ {THRESHOLDS[-1]:.4f}, 간격 {THRESHOLDS[1] - THRESHOLDS[0]:.4f}"
)

선택 모델: Stacking_LR
반복 OOF 확률: (3, 3130)
마스킹 세트: 10개
임계값: 0.4800 ~ 0.5200, 간격 0.0025


### 해석

- 4단계에서 이미 생성한 Stacking_LR 확률을 재사용하므로 모델을 다시 학습하지 않는다.
- 세 가지 Fold 시드의 OOF 확률과 마스킹 10세트를 그대로 유지한다.
- Test 확률은 아티팩트에 보관돼 있지만 이 비교에서는 읽지 않는다.

## 1. 임계값별 분류 지표 계산 함수

In [2]:
METRIC_NAMES = (
    "accuracy",
    "precision",
    "recall",
    "f1",
    "specificity",
    "fpr",
    "tn",
    "fp",
    "fn",
    "tp",
)


def calculate_threshold_metrics(y_true, probability, threshold):
    prediction = (probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp)
    return {
        "accuracy": accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0),
        "f1": f1_score(y_true, prediction, zero_division=0),
        "specificity": specificity,
        "fpr": 1 - specificity,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


print("임계값별 분류 지표 계산 함수를 준비했습니다.")

임계값별 분류 지표 계산 함수를 준비했습니다.


### 해석

임계값을 높이면 일반적으로 Won 판정이 줄어 FP와 Recall이 함께 낮아지고, 임계값을 낮추면 Won 판정이 늘어 Recall과 FP가 함께 높아진다. 실제 변화량은 저장된 OOF 확률을 기준으로 직접 확인한다.

## 2. 0.5 주변 임계값 미세 비교

In [3]:
mask_set_names = np.unique(cv_mask_set_labels)
threshold_rows = []

for threshold in THRESHOLDS:
    repeat_summaries = []
    changed_rows = []

    for probability in cv_probabilities:
        mask_set_metrics = []
        for mask_set_name in mask_set_names:
            mask = cv_mask_set_labels == mask_set_name
            mask_set_metrics.append(
                calculate_threshold_metrics(cv_target[mask], probability[mask], threshold)
            )
            baseline_prediction = probability[mask] >= BASELINE_THRESHOLD
            threshold_prediction = probability[mask] >= threshold
            changed_rows.append(np.count_nonzero(baseline_prediction != threshold_prediction))

        repeat_summaries.append(pd.DataFrame(mask_set_metrics)[list(METRIC_NAMES)].mean())

    repeat_summary = pd.DataFrame(repeat_summaries)
    threshold_rows.append(
        {
            "threshold": threshold,
            **{f"cv_{metric}_mean": repeat_summary[metric].mean() for metric in METRIC_NAMES},
            "cv_accuracy_std": repeat_summary["accuracy"].std(ddof=0),
            "cv_f1_std": repeat_summary["f1"].std(ddof=0),
            "changed_rows_mean": np.mean(changed_rows),
        }
    )

threshold_comparison = pd.DataFrame(threshold_rows)
baseline_row = threshold_comparison.loc[
    np.isclose(threshold_comparison["threshold"], BASELINE_THRESHOLD)
].iloc[0]
for metric in ("accuracy", "precision", "recall", "f1", "fp", "fn"):
    threshold_comparison[f"delta_{metric}"] = (
        threshold_comparison[f"cv_{metric}_mean"] - baseline_row[f"cv_{metric}_mean"]
    )

threshold_columns = [
    "threshold",
    "cv_accuracy_mean",
    "delta_accuracy",
    "cv_precision_mean",
    "delta_precision",
    "cv_recall_mean",
    "delta_recall",
    "cv_f1_mean",
    "delta_f1",
    "cv_specificity_mean",
    "cv_fpr_mean",
    "cv_fp_mean",
    "delta_fp",
    "cv_fn_mean",
    "delta_fn",
    "changed_rows_mean",
    "cv_accuracy_std",
    "cv_f1_std",
]
display(threshold_comparison[threshold_columns].round(6))

,threshold,cv_accuracy_mean,delta_accuracy,cv_precision_mean,delta_precision,cv_recall_mean,delta_recall,cv_f1_mean,delta_f1,cv_specificity_mean,cv_fpr_mean,cv_fp_mean,delta_fp,cv_fn_mean,delta_fn,changed_rows_mean,cv_accuracy_std,cv_f1_std
0,0.4800,0.720234,-0.000639,0.691382,-0.003782,0.812159,0.009224,0.746816,0.001743,0.625325,0.374675,57.700000,1.666667,29.866667,-1.466667,3.133333,0.000916,0.002648
1,0.4825,0.720234,-0.000639,0.691791,-0.003373,0.810901,0.007966,0.746516,0.001444,0.626623,0.373377,57.500000,1.466667,30.066667,-1.266667,2.733333,0.000543,0.002527
2,0.4850,0.719915,-0.000958,0.691895,-0.003268,0.809434,0.006499,0.745952,0.000880,0.627489,0.372511,57.366667,1.333333,30.300000,-1.033333,2.366667,0.000988,0.002997
3,0.4875,0.719915,-0.000958,0.692281,-0.002882,0.808176,0.005241,0.745653,0.000581,0.628788,0.371212,57.166667,1.133333,30.500000,-0.833333,1.966667,0.001233,0.003164
4,0.4900,0.720234,-0.000639,0.692987,-0.002176,0.807128,0.004193,0.745620,0.000548,0.630519,0.369481,56.900000,0.866667,30.666667,-0.666667,1.533333,0.001054,0.002847
5,0.4925,0.720128,-0.000745,0.693330,-0.001833,0.805660,0.002725,0.745185,0.000113,0.631818,0.368182,56.700000,0.666667,30.900000,-0.433333,1.100000,0.002488,0.004203
6,0.4950,0.720447,-0.000426,0.693864,-0.001300,0.805241,0.002306,0.745310,0.000237,0.632900,0.367100,56.533333,0.500000,30.966667,-0.366667,0.866667,0.001826,0.003663
7,0.4975,0.720554,-0.000319,0.694422,-0.000741,0.803983,0.001048,0.745095,0.000023,0.634416,0.365584,56.300000,0.266667,31.166667,-0.166667,0.433333,0.001832,0.003669
8,0.5000,0.720873,0.000000,0.695164,0.000000,0.802935,0.000000,0.745072,0.000000,0.636147,0.363853,56.033333,0.000000,31.333333,0.000000,0.000000,0.002613,0.004241
9,0.5025,0.720767,-0.000106,0.695382,0.000219,0.801887,-0.001048,0.744748,-0.000324,0.637013,0.362987,55.900000,-0.133333,31.500000,0.166667,0.300000,0.002087,0.003901


### 해석 기준

- `delta_*`는 임계값 `0.5000` 대비 변화량이다.
- `delta_fp`가 음수면 FP가 감소했고, `delta_fn`이 양수면 그만큼 Won 거래를 더 놓친 것이다.
- `changed_rows_mean`이 작으면 임계값을 바꿔도 실제 판정이 거의 달라지지 않은 것이다.
- Accuracy만 가장 높은 값을 고르지 않고 FP 감소와 FN 증가를 함께 확인한다.
- Test 10세트는 임계값을 다시 선택하는 데 사용하지 않는다.

In [4]:
assert len(threshold_comparison) == 17
assert threshold_comparison["threshold"].is_monotonic_increasing
assert threshold_comparison.notna().all().all()
assert np.isclose(baseline_row["threshold"], 0.5)
assert np.isclose(
    threshold_comparison.loc[
        np.isclose(threshold_comparison["threshold"], 0.5),
        [column for column in threshold_comparison if column.startswith("delta_")],
    ].to_numpy(),
    0,
).all()
print("5단계 임계값 민감도 검증을 통과했습니다.")

5단계 임계값 민감도 검증을 통과했습니다.


### 최종 판단

`0.5000` 주변의 미세 조정은 FP 감소만큼 FN 증가와 Recall 손실을 만들었고, Accuracy와 F1도 일관되게 개선하지 않았다. 따라서 분류 임계값은 `0.5000`으로 유지한다. Test 10세트 결과는 4단계의 분리 평가값을 그대로 보고하며 임계값 재선택에는 사용하지 않는다.